### Preprocessing of Data

In [1]:
#converting to lower case 
import pandas as pd 
df=pd.read_csv('IMDB Dataset.csv')
df['review']=df['review'].str.lower()

In [2]:
#removing of url
import re 
def remove_url(text):
    text=re.sub(r'https\S+','',text)
    return text 
df['review']=df['review'].apply(remove_url)

In [3]:
def remove_punctuation(text):
    text=re.sub(r'[^A-Za-z0-9\s]','',text)
    return text 
df['review']=df['review'].apply(remove_punctuation)

### removing html patterns 

In [4]:
def remove_html(text):
    text=re.sub(r'<.*?>','',text)
    return text 
df['review']=df['review'].apply(remove_html)

In [5]:
import nltk 
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/aayannoori/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/aayannoori/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/aayannoori/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [6]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [7]:
def remove_stopwords(text):
    token=word_tokenize(text)
    stop_w=stopwords.words('english')
    for words in token:
        if words in stop_w:
            text=text.replace(words,'')
    return text 
df['review']=df['review'].apply(remove_stopwords)

In [8]:
df.head()

,review,sentiment
0,e revewers nted wtchg 1 oz epode ll ho...,positive
1,wderful ltle producti br br filming techniqu...,positive
2,thought ths wderful wy spend tme o hot s...,positive
3,bsclly res fmly lttle boy jke thks res zom...,negative
4,petter mtte love time mey vully stunng fi...,positive


### stemming 

In [9]:
from nltk.stem import PorterStemmer 
def stemming(text):
    ps=PorterStemmer()
    stemmed_words=[]
    token=word_tokenize(text)
    for tokens in token:
        stemmed_words.append(ps.stem(tokens))
    return ' '.join(stemmed_words)  

In [10]:
df['review']=df['review'].apply(stemming)

### Encoding 

In [11]:
from sklearn.preprocessing import LabelEncoder 
le=LabelEncoder()
df['sentiment']=le.fit_transform(df['sentiment'])
y=df['sentiment']

### vectorisation

In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf=TfidfVectorizer(max_features=5000)
x=tf.fit_transform(df['review'])

### Data set and dataloader 

In [13]:
from sklearn.model_selection import train_test_split 
import torch 
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
from torch.utils.data import TensorDataset,DataLoader
x_train=x_train.toarray()
x_test=x_test.toarray()
train_set=TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.from_numpy(y_train.values).float(),  
)
test_set=TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.from_numpy(y_test.values).float(),  
)
train_loader=DataLoader(train_set,shuffle=True,batch_size=64)
test_loader=DataLoader(test_set,shuffle=True,batch_size=64)

### Building RNN 

In [14]:
import torch
import torch.nn as nn

class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super(RNN, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:,-1,:])
        return out

In [15]:
import torch.optim as optim 
input_layer=x_train.shape[1]
model=RNN(input_layer)
criterion=nn.BCELoss()
optimiser=optim.Adam(model.parameters())

In [19]:
best_val_loss = float("inf")
nums_of_epoch=10
for epoch in range(nums_of_epoch):
    model.train()
    for xb, yb in train_loader:
        optimiser.zero_grad()
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        probs = torch.sigmoid(outputs.squeeze())
        loss = criterion(probs, yb)
        loss.backward()
        optimiser.step()
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.unsqueeze(1)
            outputs = model(xb)
            probs = torch.sigmoid(outputs.squeeze())
            loss = criterion(probs, yb)
            val_loss += loss.item()
    val_loss /= len(test_loader)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_rnn_weights.pth")

    print(f"epoch {epoch+1} | val_loss={val_loss:.4f} | best={best_val_loss:.4f}")


epoch 1 | val_loss=0.3097 | best=0.3097
epoch 2 | val_loss=0.3228 | best=0.3097
epoch 3 | val_loss=0.3246 | best=0.3097
epoch 4 | val_loss=0.3275 | best=0.3097
epoch 5 | val_loss=0.3372 | best=0.3097
epoch 6 | val_loss=0.3340 | best=0.3097
epoch 7 | val_loss=0.3394 | best=0.3097
epoch 8 | val_loss=0.3392 | best=0.3097
epoch 9 | val_loss=0.3417 | best=0.3097
epoch 10 | val_loss=0.3412 | best=0.3097


In [20]:
import pickle

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tf, f)
